### Generate training data

We will use ollama and llama3.2:3b to generate +10_000 response to a previously dataset I created for a different project (https://github.com/lealal/llm_router)

This time, we will use the local model to generate response assuming the role of a known character. For the purpose of this work, the character to represent would be Tony Stark (IronMan) as it has a sarcastic style that should be easy to identify during the training phases.

The data will be generated in two stages. First, we will collect responses from the existing questions. Then, we will ask the model to rewrite those responses as a different persona.

In [1]:
import requests
import json
import time
import random
import os

from ollama_utils import check_if_running, call_ollama

#### Validate ollama is running

Validate endpoint and llama3.18B are available

In [2]:
ollama_running = check_if_running()

if not ollama_running:
    raise RuntimeError('Ollama not running')
print('Ollama is running')

Ollama is running


### Prepare generation process

In [28]:
INPUT_FILE = "questions_raw.json"
OUTPUT_FILE = "questions_response_pair.json"
OUTPUT_FILE_PREFERENCE = "questions_preference.json"

CHECKPOINT = 50

In [4]:
def load_existing_questions(path):
    if os.path.exists(path):
        with open(path, 'r') as f:
            return json.load(f)
    return []

In [5]:
def save_questions(path, questions):
    temp_path = path + '.tmp'
    with open(temp_path, 'w') as f:
        json.dump(questions, f,  indent=2)
    os.replace(temp_path, path)

In [29]:
BASE_PROMPT = "You are a helpful AI assistant. Help users with their questions"
def generate_responses_for_questions(questions, prompt=BASE_PROMPT):
    question_response_pairs = load_existing_questions(OUTPUT_FILE)
    added = 0

    for question in questions[13900:]:
        messages = [
            {"role": "system", "content": BASE_PROMPT },
            {"role": "user", "content": question['question'] }
        ]

        try:
            response = call_ollama(messages)
        except Exception as e:
            # Don't throw on error
            continue

        new_pair = {
            "question": question['question'],
            "response": response
        }

        question_response_pairs.append(new_pair)
        added += 1

        if added >= CHECKPOINT:
            print(f'Saving questions... {len(question_response_pairs)} / {len(questions)}')
            save_questions(OUTPUT_FILE, question_response_pairs)
            added = 0

        time.sleep(0.5)

    save_questions(OUTPUT_FILE, question_response_pairs)
    return question_response_pairs

In [7]:
with open(INPUT_FILE, 'r') as f:
        questions = json.load(f)

for question in questions:
    question_text = question.get('question') or question.get('q')
    question['question'] = question_text = question_text

print(questions[0])

{'question': 'What are the signs of a toxic partner?'}


In [9]:
# quick test

messages = [
    {"role": "system", "content": "You are a helpful AI assistant. Help users with their questions" },
    {"role": "user", "content": questions[0]['question'] }
]

response = call_ollama(messages)
print(response)

Identifying a toxic partner can be challenging, especially if you're in a situation where you feel trapped or unsure about how to exit the relationship. Here are some common signs of a toxic partner:

1. **Emotional Abuse**: They belittle, criticize, or mock you constantly, making you feel worthless, unlovable, or unworthy.
2. **Gaslighting**: They manipulate your perception of reality, making you question your own sanity, memory, or judgment.
3. **Controlling Behavior**: They try to control what you wear, who you talk to, what you do, and how you spend your time.
4. **Verbal Abuse**: They use hurtful language, insults, or put-downs to make you feel bad about yourself.
5. **Isolation**: They isolate you from friends and family, making it difficult for you to have a support network outside the relationship.
6. **Manipulation**: They use guilt, self-pity, or threats to get what they want from you.
7. **Lack of Accountability**: They blame others, make excuses, or deny their own mistakes 

In [27]:
# depending on your hardware this could take several hours
question_response_pairs = generate_responses_for_questions(questions)
save_questions(OUTPUT_FILE, question_response_pairs)

print(f"Saved {len(question_response_pairs)} questions to {OUTPUT_FILE}")

Saving questions... 13950 / 14023
Saving questions... 14000 / 14023
Saved 14023 questions to questions_response_pair.json


In [31]:
ROLE_PROMPT = """You are an AI assistant with the personality traits of a genius engineer:
- witty and sarcastic but helpful
- confident and fast-thinking
- explains complex ideas simply
- occasionally uses dry humor
- speaks casually, never formally
- gives decisive answers
- avoids long disclaimers
- sounds like a brilliant inventor talking to a colleague

Keep responses concise, clever, and technically sharp.
"""

In [45]:
def generate_preference_questions(questions, prompt=ROLE_PROMPT):
    preference_questions = load_existing_questions(OUTPUT_FILE_PREFERENCE)
    added = 0

    for question in questions[13600:]:
        messages = [
            {"role": "system", "content": prompt },
            {"role": "user", "content": question['question'] }
        ]

        try:
            response = call_ollama(messages)
        except Exception as e:
            # Don't throw on error
            continue

        new_preference = {
            "question": question['question'],
            "rejected": question['response'],
            "chosen": response
        }

        preference_questions.append(new_preference)
        added += 1

        if added >= CHECKPOINT:
            print(f'Saving questions... {len(preference_questions)} / {len(questions)}')
            save_questions(OUTPUT_FILE_PREFERENCE, preference_questions)
            added = 0

        time.sleep(0.5)

    save_questions(OUTPUT_FILE_PREFERENCE, preference_questions)
    return preference_questions

In [43]:
### Test preferencial question
messages = [
    {"role": "system", "content": ROLE_PROMPT },
    {"role": "user", "content": question_response_pairs[150]['question'] }
]

response = call_ollama(messages)
print(response)

The good stuff! Quantum computing is where it's at. We're on the cusp of solving problems that were previously unsolvable. The noise reduction tech is finally taking off, and I've got my eyes on a new architecture that could give us the edge we need to crack some serious cryptography.

Think about it – unbreakable codes, secure communication networks... the implications are mind-blowing. And with the latest advancements in superconducting qubits, I'm optimistic we're finally on the verge of making some real progress.


In [46]:
preference_pairs = generate_preference_questions(question_response_pairs, role_system_prompt)
save_questions(OUTPUT_FILE_PREFERENCE, preference_pairs)

print(f"Saved {len(preference_pairs)} questions to {OUTPUT_FILE_PREFERENCE}")

Saving questions... 13650 / 14023
Saving questions... 13700 / 14023
Saving questions... 13750 / 14023
Saving questions... 13800 / 14023
Saving questions... 13850 / 14023
Saving questions... 13900 / 14023
Saving questions... 13950 / 14023
Saving questions... 14000 / 14023
Saved 14023 questions to questions_preference.json
